# 09 · Baseline & broadband-inflation check

Supports the report's language-area caveat: during the interview *all* bands (incl. alpha)
inflate together, so %-changes vs rest are not interpretable (left/right asymmetry is).

In [1]:
import warnings, pathlib, sys
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
_c = pathlib.Path.cwd()
REPO = next(p for p in [_c] + list(_c.parents) if (p / "analysis").is_dir())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from analysis import dataset, eegkit as K
DATA_DIR = None
recs = dataset.discover(DATA_DIR)
_gi, LABEL = {}, {}
for _r in recs:
    _g = _r["group"]; _gi[_g] = _gi.get(_g, 0) + 1
    LABEL[_r["key"]] = ("exp" if _g == "E" else "ctrl") + str(_gi[_g])
def lbl(r): return LABEL[r["key"]]
FIG = REPO / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
def ec_window(r):
    st = K.get_stages(K.load(r["path"]), r["key"])
    return [s for s in st if s[0].startswith("rest_close")][0]
def get_iaf(r):
    raw = K.load(r["path"]); ec = ec_window(r)
    _, live = K.get(raw, ec[1], ec[2]); return K.iaf(raw, live, ec[1], ec[2])
def stage_of(r, nm):
    st = K.get_stages(K.load(r["path"]), r["key"]); return [s for s in st if s[0] == nm]
print(f"discovered {len(recs)} recordings ->", {lbl(r): r["group"] for r in recs})


discovered 4 recordings -> {'exp1': 'E', 'exp2': 'E', 'ctrl1': 'C', 'ctrl2': 'C'}


In [2]:
FT = ("F7","F3","T3","T5","F4","F8","T4","T6")
for r in recs:
    raw=K.load(r["path"]); fs=raw.info["sfreq"]; IAF=get_iaf(r)
    st=K.get_stages(raw, r["key"])
    print("="*70); print(f"{lbl(r)}  IAF {IAF:.1f}   band levels per stage (uV^2, frontal+temporal avg):")
    print(f"{'stage':<13}{'muscle30-80':>12}{'delta':>9}{'alpha':>8}{'fmtheta':>9}")
    for name,a,b in st:
        x,live=K.get(raw,a,b)
        ii=[live.index(c) for c in FT if c in live]
        mus=K.band_pow(x[ii],fs,30,80).mean(); dl=K.band_pow(x[ii],fs,1,4).mean()
        al=K.band_pow(x[ii],fs,IAF-2,IAF+2).mean()
        fz=live.index("Fz") if "Fz" in live else live.index("FCz")
        th=K.band_pow(x[fz],fs,4,8).mean()
        print(f"{name:<13}{mus:>12.1f}{dl:>9.1f}{al:>8.1f}{th:>9.1f}")
print("\nSpeech rows have muscle+delta+alpha all high -> broadband inflation, not clean ERD.")


exp1  IAF 9.6   band levels per stage (uV^2, frontal+temporal avg):
stage         muscle30-80    delta   alpha  fmtheta


rest_open1           48.8     83.6     3.8      5.8


rest_close1          46.5     41.5    38.1      5.4


hypervent            44.1    113.3     8.3     10.4


rest_open2           61.1     98.3     8.9      7.9


...

rest_open3           14.5     11.2     3.5      4.0


speech               33.0     17.3     4.1      3.9


ctrl2  IAF 9.9   band levels per stage (uV^2, frontal+temporal avg):
stage         muscle30-80    delta   alpha  fmtheta


speech               60.7     23.2     4.8      4.5
rest_open4           11.7     11.1     4.6      3.5


rest_close4           5.4      6.9    16.0      4.1

Speech rows have muscle+delta+alpha all high -> broadband inflation, not clean ERD.
